<a href="https://colab.research.google.com/github/ned1313/Fine-tuning-and-Optimizing-Small-Language-Models/blob/main/SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install transformers datasets accelerate peft trl bitsandbytes pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
import gc
import io
import json
import time
from contextlib import redirect_stderr, redirect_stdout
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from transformers.utils import logging as transformers_logging
from trl import SFTConfig, SFTTrainer

try:
    import pynvml
except Exception:
    pynvml = None

set_seed(42)
transformers_logging.set_verbosity_error()

if not torch.cuda.is_available():
    raise RuntimeError("This notebook is intended to run on a CUDA GPU.")

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
history_path = Path("./benchmark_history.jsonl")


def pick_attention_implementation() -> str:
    try:
        import flash_attn  # noqa: F401
    except Exception:
        return "sdpa"
    return "flash_attention_2"


attention_implementation = pick_attention_implementation()
device = torch.device("cuda")

print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Compute dtype: {compute_dtype}")
print(f"Attention implementation: {attention_implementation}")
print(f"History file: {history_path.resolve()}")


def clear_gpu_memory() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def load_benchmark_history(path: Path = history_path) -> list[dict]:
    if not path.exists():
        return []
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as file_obj:
        for line in file_obj:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


def append_benchmark_history(new_rows: list[dict], path: Path = history_path) -> None:
    if not new_rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as file_obj:
        for row in new_rows:
            file_obj.write(json.dumps(row) + "\n")


def get_peak_gpu_utilization_pct() -> float | None:
    if pynvml is None:
        return None
    try:
        if not getattr(get_peak_gpu_utilization_pct, "_initialized", False):
            pynvml.nvmlInit()
            get_peak_gpu_utilization_pct._initialized = True
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        utilization = pynvml.nvmlDeviceGetUtilizationRates(handle)
        return round(float(utilization.gpu), 1)
    except Exception:
        return None


def get_trainable_parameter_counts(model: torch.nn.Module) -> dict[str, int]:
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    total = sum(parameter.numel() for parameter in model.parameters())
    return {"trainable": trainable, "total": total}



CUDA device: Tesla T4
Compute dtype: torch.bfloat16
Attention implementation: sdpa
History file: /content/benchmark_history.jsonl


In [3]:
model_id = "Qwen/Qwen3-0.6B-Base"
dataset_name = "openai/gsm8k"
dataset_config = "main"
train_examples = 64
eval_examples = 8

raw_dataset = load_dataset(dataset_name, dataset_config)


def format_gsm8k_example(example: dict) -> dict:
    question = example["question"].strip()
    answer = example["answer"].strip()
    prompt = (
        "Solve the following grade-school math problem.\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )
    return {"text": f"{prompt} {answer}"}


train_dataset = raw_dataset["train"].shuffle(seed=42).select(range(train_examples)).map(
    format_gsm8k_example,
    remove_columns=raw_dataset["train"].column_names,
)

eval_dataset = raw_dataset["test"].shuffle(seed=42).select(range(eval_examples)).map(
    format_gsm8k_example,
    remove_columns=raw_dataset["test"].column_names,
)

print(train_dataset[0]["text"][:500])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Solve the following grade-school math problem.

Question: Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?

Answer: Mimi has 2 x 12 = <<2*12=24>>24 sea shells.
Kyle has 24 x 2 = <<24*2=48>>48 sea shells.
Leigh has 48 / 3 = <<48/3=16>>16 sea shells.
#### 16


In [4]:
def make_lora_config() -> LoraConfig:
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    )


def build_model(strategy: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if strategy == "qlora":
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            trust_remote_code=True,
            device_map="auto",
            quantization_config=quantization_config,
            torch_dtype=compute_dtype,
            attn_implementation=attention_implementation,
        )
        model = prepare_model_for_kbit_training(model)
        model = get_peft_model(model, make_lora_config())
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            trust_remote_code=True,
            torch_dtype=compute_dtype,
            attn_implementation=attention_implementation,
        ).to(device)
        if strategy == "lora":
            model = get_peft_model(model, make_lora_config())

    model.config.use_cache = False
    return model, tokenizer


def run_experiment(strategy: str, batch_size: int, sequence_length: int, max_steps: int = 8) -> dict:
    clear_gpu_memory()
    model, tokenizer = build_model(strategy)
    counts = get_trainable_parameter_counts(model)

    if strategy == "full":
        learning_rate = 2e-5
        optimizer = "adamw_torch"
    elif strategy == "lora":
        learning_rate = 1e-4
        optimizer = "adamw_torch"
    else:
        learning_rate = 1e-4
        optimizer = "paged_adamw_8bit"

    train_args = SFTConfig(
        output_dir=f"./outputs/{strategy}",
        max_length=sequence_length,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        max_steps=max_steps,
        logging_steps=max_steps,
        save_strategy="no",
        report_to=[],
        bf16=compute_dtype == torch.bfloat16,
        fp16=compute_dtype == torch.float16,
        gradient_checkpointing=True,
        learning_rate=learning_rate,
        optim=optimizer,
        disable_tqdm=True,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        args=train_args,
        processing_class=tokenizer,
        formatting_func=lambda examples: examples["text"],
    )

    start = time.perf_counter()
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        trainer.train()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    peak_allocated_mib = round(torch.cuda.max_memory_allocated() / 1024**2, 1)
    peak_reserved_mib = round(torch.cuda.max_memory_reserved() / 1024**2, 1)
    peak_gpu_util_pct = get_peak_gpu_utilization_pct()
    print(f"{strategy}: {max_steps} steps, {elapsed:.2f}s, peak alloc {peak_allocated_mib} MiB")

    result = {
        "strategy": strategy,
        "batch_size": batch_size,
        "sequence_length": sequence_length,
        "max_steps": max_steps,
        "peak_allocated_mib": peak_allocated_mib,
        "peak_reserved_mib": peak_reserved_mib,
        "peak_gpu_util_pct": peak_gpu_util_pct,
        "approx_tokens_per_second": round((batch_size * sequence_length * max_steps) / elapsed, 1),
        "trainable_parameters": counts["trainable"],
        "total_parameters": counts["total"],
    }

    del trainer, model, tokenizer
    gc.collect()
    clear_gpu_memory()
    return result

In [6]:
batch_size = 1
sequence_length = 2048
max_steps = 24

strategies = ["full"]

if "benchmark_history" not in globals():
    benchmark_history = load_benchmark_history()

if "run_number" not in globals():
    prior_run_numbers = [row.get("run_number", 0) for row in benchmark_history if isinstance(row, dict)]
    run_number = max(prior_run_numbers, default=0) + 1

results = []
for strategy in strategies:
    result = run_experiment(
        strategy,
        batch_size=batch_size,
        sequence_length=sequence_length,
        max_steps=max_steps,
    )
    result["run_number"] = run_number
    results.append(result)

benchmark_history.extend(results)
append_benchmark_history(results)
run_number += 1

current_results_df = pd.DataFrame(results)
current_results_df = current_results_df[
    [
        "run_number",
        "strategy",
        "batch_size",
        "sequence_length",
        "max_steps",
        "peak_allocated_mib",
        "peak_reserved_mib",
        "peak_gpu_util_pct",
        "approx_tokens_per_second",
    ]
]

history_df = pd.DataFrame(benchmark_history)
history_df = history_df[
    [
        "run_number",
        "strategy",
        "batch_size",
        "sequence_length",
        "max_steps",
        "peak_allocated_mib",
        "peak_reserved_mib",
        "peak_gpu_util_pct",
        "approx_tokens_per_second",
    ]
]
history_df = history_df.sort_values(["run_number", "strategy"]).reset_index(drop=True)

display(current_results_df)
display(history_df.tail(24))

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

full: 24 steps, 16.04s, peak alloc 5737.6 MiB


,run_number,strategy,batch_size,sequence_length,max_steps,peak_allocated_mib,peak_reserved_mib,peak_gpu_util_pct,approx_tokens_per_second
0,2,full,1,2048,24,5737.6,6362.0,98.0,3064.2


,run_number,strategy,batch_size,sequence_length,max_steps,peak_allocated_mib,peak_reserved_mib,peak_gpu_util_pct,approx_tokens_per_second
0,1,full,8,2048,24,5779.9,6988.0,97.0,3168.4
1,2,full,1,2048,24,5737.6,6362.0,98.0,3064.2
